Python Privacy Gateway & Token Vault (Redis + PostgreSQL) คือสถาปัตยกรรมระบบรักษาความปลอดภัยของข้อมูล (Data Privacy Architecture) ที่พัฒนาด้วย Python เพื่อทำหน้าที่เป็น "ด่านหน้า" คัดกรอง ขจัด หรือแปลงข้อมูลระบุตัวตน (PII - Personally Identifiable Information เช่น เลขบัตรประชาชน บัตรเครดิต เบอร์โทรศัพท์ หรืออีเมล) ให้อยู่ในรูปของ Token (รหัสสุ่มที่ไม่สื่อความหมาย) ก่อนส่งข้อมูลต่อไปยังระบบภายนอกหรือฐานข้อมูลหลัก

Privacy Gateway (Python)

ทำหน้าที่เป็น Reverse Proxy หรือ Middleware (สร้างด้วย Framework เช่น FastAPI หรือ Flask) ดักจับ Request/Response , สแกนตรวจจับข้อมูล PII โดยใช้ Regex, กฎเฉพาะทาง หรือโมเดล NLP/NER (เช่น Microsoft Presidio) , สลับข้อมูล PII จริงด้วย Token ก่อนส่งเข้าแอปพลิเคชันภายใน หรือสลับ Token คืนเป็นข้อมูลจริงเมื่อต้องส่งให้ผู้ให้บริการที่ได้รับอนุญาต

[ Client / Third-party ]
       │
       ▼ (Request พร้อม PII จริง)
[ FastAPI Privacy Gateway (Reverse Proxy / Middleware) ]
       │
       ├── 1. สแกน PII (Presidio NER + Custom Regex เช่น Thai ID, Phone, Email)
       ├── 2. Tokenize: สร้าง tokenUUID สุ่ม
       └── 3. บันทึกลง Token Vault:
              ├── Redis (แคช Token <-> Plaintext ชั่วคราว / Sub-millisecond)
              └── PostgreSQL (เข้ารหัส AES/Fernet + Audit Log ถาวร)
       │
       ▼ (ส่ง Payload ที่ถูก Tokenized แล้ว เข้า Application ภายใน)
[ Internal Backend Service / LLM Service ]
       │
       ▲ (Response ส่งกลับ หรือ Outbound Request ไป Partner)
[ Detokenization Layer ] ──> สลับ Token คืนเป็นข้อมูลจริง (เฉพาะผู้มีสิทธิ์)

Token Vault (ทำหน้าที่จัดเก็บคู่ Token <-> Plaintext)

Redis (In-Memory Hot Cache): จัดเก็บคู่แมปปิ้ง Token <-> ข้อมูลจริง แบบมีอายุ (TTL) สำหรับการค้นหาและแปลงกลับแบบเรียลไทม์ (Sub-millisecond latency) , PostgreSQL (Persistent & Encrypted Storage): บันทึกคู่ Token และข้อมูลจริงที่ผ่านการเข้ารหัสระดับฟิลด์ (Envelope Encryption หรือ AES-256-GCM) ไว้อย่างถาวร พร้อมจัดเก็บ Audit Log ว่าใครเข้าถึงหรือเรียกคืนข้อมูลเมื่อใด

In [ ]:
import re
import json
import uuid
import hashlib
import threading
import httpx
import uvicorn
import nest_asyncio
from typing import List, Dict, Optional
from dataclasses import dataclass
from fastapi import FastAPI, Request, Response
from cryptography.fernet import Fernet

# สำหรับรัน asyncio ใน Jupyter
nest_asyncio.apply()

try:
    from presidio_analyzer import AnalyzerEngine
    PRESIDIO_AVAILABLE = True
    print("Presidio ready")
except ImportError:
    PRESIDIO_AVAILABLE = False
    print("Presidio not available, will use regex fallback")

PII Scanner Engine (pii_scanner.py)
ใช้ Microsoft Presidio สำหรับสแกน PII สากล (Email, Phone, Credit Card, IP Address, Person) พร้อมลงทะเบียน Custom Pattern (Regex) เช่น เลขบัตรประจำตัวประชาชนไทย (13 หลัก)

In [ ]:
import re
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from vault import TokenVault

class PIIScanner:
    def __init__(self):
        self.analyzer = AnalyzerEngine()
        self._register_thai_id_recognizer()

    def _register_thai_id_recognizer(self):
        """เพิ่ม Custom Regex สำหรับตรวจจับเลขประจำตัวประชาชนไทย 13 หลัก"""
        thai_id_pattern = Pattern(
            name="thai_id_pattern",
            regex=r"\b\d{1}[-\s]?\d{4}[-\s]?\d{5}[-\s]?\d{2}[-\s]?\d{1}\b",
            score=0.85
        )
        thai_id_recognizer = PatternRecognizer(
            supported_entity="THAI_CITIZEN_ID",
            patterns=[thai_id_pattern]
        )
        self.analyzer.registry.add_recognizer(thai_id_recognizer)

    def mask_and_tokenize(self, text: str) -> str:
        """ค้นหา PII ทั้งหมดในข้อความ แล้วแปลงเป็น Token ผ่าน Vault"""
        if not text:
            return text

        results = self.analyzer.analyze(
            text=text,
            language="en",
            entities=["EMAIL_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD", "PERSON", "THAI_CITIZEN_ID"]
        )

        # เรียงลำดับจากท้ายมาหน้าเพื่อไม่ให้ offset ตำแหน่งของ index เปลี่ยนตอนแทนที่
        sorted_results = sorted(results, key=lambda x: x.start, reverse=True)
        sanitized_text = text

        for entity in sorted_results:
            raw_value = text[entity.start:entity.end]
            # ตัดช่องว่างหรือขีดออกกรณีเป็นตัวเลข
            clean_value = raw_value.strip()
            
            # ส่งเข้า Token Vault เพื่อสร้าง/ดึง Token
            token = TokenVault.tokenize(real_value=clean_value, pii_type=entity.entity_type)
            
            # แทนที่ข้อมูลจริงด้วย Token
            sanitized_text = sanitized_text[:entity.start] + token + sanitized_text[entity.end:]

        return sanitized_text

    def restore_tokens(self, text: str) -> str:
        """สแกนหา Token pattern (tok_...) แล้วแปลงคืนเป็น Plaintext"""
        token_regex = r"\btok_[a-z_]+_[a-f0-9]{12}\b"
        
        def replace_token(match):
            token = match.group(0)
            real_val = TokenVault.detokenize(token)
            return real_val if real_val else token

        return re.sub(token_regex, replace_token, text)

Privacy Gateway

In [ ]:
import json
import httpx
from fastapi import FastAPI, Request, Response, HTTPException, Header, Depends
from pii_scanner import PIIScanner

app = FastAPI(title="Privacy Gateway & Token Vault")
scanner = PIIScanner()

# ปลายทาง Internal Backend Service หรือ LLM Cluster
TARGET_UPSTREAM_URL = "http://localhost:8080"
AUTHORIZED_ADMIN_API_KEY = "super-secret-gateway-key"


def verify_detokenize_permission(x_api_key: str = Header(None)):
    """ตรวจสอบสิทธิ์ก่อนอนุญาตให้คืนค่า PII จริง"""
    if x_api_key != AUTHORIZED_ADMIN_API_KEY:
        raise HTTPException(status_code=403, detail="Unauthorized to detokenize sensitive PII")
    return True


@app.api_route("/proxy/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "PATCH"])
async def reverse_proxy_gateway(request: Request, path: str):
    """
    Reverse Proxy:
    ดัก Request -> สแกน PII -> แทนที่ด้วย Token -> Forward ไป Internal Service
    """
    upstream_url = f"{TARGET_UPSTREAM_URL}/{path}"
    headers = dict(request.headers)
    headers.pop("host", None)  # ป้องกันปัญหา host mismatch

    # อ่าน Body ของ request
    body = await request.body()
    transformed_body = body

    if body:
        try:
            # กรณีเป็น JSON: แปลง JSON สแกน PII แบบ recursive
            json_data = json.loads(body.decode())

            def sanitize_obj(obj):
                if isinstance(obj, str):
                    return scanner.mask_and_tokenize(obj)
                elif isinstance(obj, dict):
                    return {k: sanitize_obj(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [sanitize_obj(i) for i in obj]
                return obj

            sanitized_json = sanitize_obj(json_data)
            transformed_body = json.dumps(sanitized_json).encode("utf-8")
            headers["content-length"] = str(len(transformed_body))
        except Exception:
            # กรณีเป็น Raw Text / ไม่ใช่ JSON
            plain_text = body.decode("utf-8", errors="ignore")
            transformed_body = scanner.mask_and_tokenize(plain_text).encode("utf-8")
            headers["content-length"] = str(len(transformed_body))

    # ส่งต่อ Request ไปยังระบบภายใน
    async with httpx.AsyncClient(timeout=30.0) as client:
        upstream_response = await client.request(
            method=request.method,
            url=upstream_url,
            headers=headers,
            params=request.query_params,
            content=transformed_body,
        )

    # ส่ง Response คืนกลับไปยัง Caller
    return Response(
        content=upstream_response.content,
        status_code=upstream_response.status_code,
        headers=dict(upstream_response.headers),
    )


@app.post("/vault/detokenize", dependencies=[Depends(verify_detokenize_permission)])
async def detokenize_payload(payload: dict):
    """
    Endpoint สำหรับ Authorized Services:
    รับข้อความหรือ JSON ที่ติด Token แล้วแปลงคืนเป็น Plaintext ข้อมูลจริง
    """
    text_content = payload.get("text", "")
    restored_text = scanner.restore_tokens(text_content)
    return {"restored_text": restored_text}


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
import uvicorn

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)  # รอเซิร์ฟเวอร์เริ่มทำงาน
print("FastAPI Gateway running at http://127.0.0.1:8000")

In [ ]:
sample_order = {
    "customer_name": "Somchai Prasert",
    "email": "somchai.p@example.com",
    "citizen_id": "1-1002-00300-40-1",
    "phone": "0891234567"
}

print("--- 1. ส่งข้อมูล PII จริงเข้าไปยัง Gateway ---")
res = requests.post("http://127.0.0.1:8000/internal/orders", json=sample_order)

print("Response จากระบบภายใน:")
print(json.dumps(res.json(), indent=2, ensure_ascii=False))

In [ ]:
tokenized_data = res.json()["processed_data"]

print("--- 2. ทดสอบถอดรหัสแบบไม่ใส่ API Key (ควรโดน 403) ---")
unauthorized_res = requests.post("http://127.0.0.1:8000/vault/detokenize", json=tokenized_data)
print("Status Code:", unauthorized_res.status_code)

print("\n--- 3. ถอดรหัสด้วยสิทธิ์ Authorized Partner ---")
headers = {"x-api-key": "super-secret-gateway-key"}
authorized_res = requests.post("http://127.0.0.1:8000/vault/detokenize", json=tokenized_data, headers=headers)

print("ข้อมูลจริงที่ถอดรหัสคืนกลับมา:")
print(json.dumps(authorized_res.json(), indent=2, ensure_ascii=False))